# Image Fundamentals Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Your first canvas.** An image IS a NumPy array: shape `(rows, cols)`, one `uint8` byte per pixel, indexed `[y, x]`.

In [ ]:
import numpy as np

img = np.zeros((100, 160), dtype=np.uint8)

print("shape:", img.shape, "| dtype:", img.dtype)
print("range:", img.min(), "..", img.max())

img[50, 100] = 255              # [y, x]: row 50, column 100
print("pixel at (x=100, y=50):", img[50, 100])

**2. Stripe orientation.** Axis 0 is y/rows, axis 1 is x/columns - slicing all rows over a column range paints a VERTICAL stripe.

In [ ]:
import numpy as np

canvas = np.zeros((80, 200), dtype=np.uint8)

canvas[:, 90:110] = 255         # ALL rows, columns 90..109 -> vertical stripe

print(canvas.shape)                          # (80 rows, 200 columns)
print("white pixels:", int((canvas == 255).sum()))   # 80 * 20 = 1600

**3. The 200 + 100 = 44 trap.** `uint8` math wraps modulo 256; OpenCV arithmetic saturates at 255 instead.

In [ ]:
import numpy as np
import cv2

base = np.full((60, 90), 200, dtype=np.uint8)

wrapped   = base + 100                                        # uint8: WRAPS
saturated = cv2.add(base, 100)                                # OpenCV: CLAMPS
manual    = np.clip(base.astype(np.int16) + 100, 0, 255).astype(np.uint8)

print("numpy uint8 :", wrapped.ravel()[0])     # 44   (!!)
print("cv2.add     :", saturated.ravel()[0])   # 255
print("clip idiom  :", manual.ravel()[0])      # 255
# 200 + 100 = 300, but uint8 holds 0..255: NumPy wraps (300 % 256 = 44);
# cv2.add saturates instead - widen, clip, cast back when doing manual math.

## Part 2 — Practice

**4. Paint a scene from nothing.** Blank canvas + drawing primitives = a reproducible fixture; colors are `(B, G, R)` and matplotlib needs RGB.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

scene = np.zeros((240, 320, 3), dtype=np.uint8)

cv2.rectangle(scene, (0, 160), (320, 240), (30, 90, 40), -1)   # grass, filled
cv2.circle(scene, (260, 55), 25, (0, 215, 255), -1)            # yellow sun (BGR!)
cv2.putText(scene, "OpenCV", (16, 36),
            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)

plt.figure(figsize=(5, 4))
plt.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))   # BGR -> RGB before display!
plt.axis("off")
plt.show()

print("scene:", scene.shape, scene.dtype)

**5. Channel surgery.** Channel 0 is blue, channel 2 is red - `img[:, :, c]` peels one plane off the stack.

In [ ]:
import numpy as np
import cv2

board = np.zeros((120, 280, 3), dtype=np.uint8)
cv2.rectangle(board, (20, 30), (120, 90), (0, 0, 255), -1)     # RED   (B=0, G=0, R=255)
cv2.rectangle(board, (160, 30), (260, 90), (255, 0, 0), -1)    # BLUE

print("red square pixel :", board[60, 70])      # [  0   0 255]
print("blue square pixel:", board[60, 210])     # [255   0   0]

blue_plane = board[:, :, 0]
red_plane = board[:, :, 2]

print(blue_plane.shape, red_plane.shape)        # both (120, 280)
print("max blue plane:", blue_plane.max(), "| max red plane:", red_plane.max())

**6. Crop with confidence.** Slicing returns a VIEW sharing memory; only `.copy()` makes a crop independent.

In [ ]:
import numpy as np

img = np.full((200, 300), 40, dtype=np.uint8)
img[30:150, 30:130] = 255               # a white block to find

patch = img[20:100, 100:180]            # [y1:y2, x1:x2], stop exclusive
print(patch.shape)                      # (80, 80)

patch[:] = 77                           # slices are VIEWS...
print("block interior now:", img[35, 110])    # 77 - the original changed!

patch2 = img[20:100, 100:180].copy()    # .copy() breaks the link
patch2[:] = 0
print("block interior after copy edit:", img[35, 110])   # still 77
# Rule: crops share memory unless you explicitly .copy() them.

**7. Resize, width first.** `resize` wants `(width, height)` - the opposite order from `shape`; INTER_AREA for shrinking, LINEAR/CUBIC/LANCZOS4 for growing.

In [ ]:
import numpy as np
import cv2

row = np.linspace(30, 220, 240).astype(np.uint8)     # one row, 240 columns
grad = np.tile(row, (120, 1))                        # repeated down 120 rows
print("original:", grad.shape)

small = cv2.resize(grad, (120, 60),                  # (WIDTH, height)!
                   interpolation=cv2.INTER_AREA)     # AREA averages when shrinking
print("shrunk  :", small.shape)

up_near = cv2.resize(small, (240, 120), interpolation=cv2.INTER_NEAREST)
up_best = cv2.resize(small, (240, 120), interpolation=cv2.INTER_LANCZOS4)
print("enlarged:", up_near.shape, up_best.shape)
# NEAREST copies pixels (fast, blocky); LANCZOS4 invents the smoothest detail.

## Part 3 — Challenge

**8. HSV fruit picker.** Hue separates color family from brightness, so an `inRange` band catches the green ball regardless of shade.

In [ ]:
import numpy as np
import cv2

basket = np.full((220, 340, 3), 35, dtype=np.uint8)     # dark tray
cv2.circle(basket, (70, 110), 46, (255, 0, 0), -1)      # BLUE ball
cv2.circle(basket, (170, 130), 46, (0, 180, 0), -1)     # GREEN ball
cv2.circle(basket, (270, 105), 46, (0, 0, 255), -1)     # RED ball

hsv = cv2.cvtColor(basket, cv2.COLOR_BGR2HSV)           # H: 0-179, S/V: 0-255
mask = cv2.inRange(hsv, (35, 80, 80), (85, 255, 255))   # green hue band
only_green = cv2.bitwise_and(basket, basket, mask=mask)

print("mask coverage: %.1f%%" % (100 * (mask > 0).mean()))
print("green pixels kept:", int((only_green[:, :, 1] > 0).sum()))
# Only the green ball falls inside hue 35-85 - blue (~110) and red (~0/wrap) miss.

**9. Round trip: PNG vs JPEG.** PNG reloads byte-for-byte; JPEG trades pixels for size - always `imwrite` then verify with `array_equal`.

In [ ]:
from pathlib import Path
import numpy as np
import cv2

Path("sample_data").mkdir(exist_ok=True)

logo = np.zeros((160, 200, 3), dtype=np.uint8)
cv2.circle(logo, (100, 70), 45, (0, 140, 255), -1)
cv2.putText(logo, "PyMastery", (28, 145),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

assert cv2.imwrite("sample_data/exercise_logo.png", logo)
assert cv2.imwrite("sample_data/exercise_logo.jpg", logo,
                   [cv2.IMWRITE_JPEG_QUALITY, 40])

png_back = cv2.imread("sample_data/exercise_logo.png")
jpg_back = cv2.imread("sample_data/exercise_logo.jpg")
assert png_back is not None and jpg_back is not None    # imread returns None on failure!

print("PNG identical :", np.array_equal(logo, png_back))
print("JPG identical :", np.array_equal(logo, jpg_back))
print("worst JPG drift:", int(np.abs(logo.astype(int) - jpg_back).max()))
# PNG is lossless -> safe for pixel-exact tests; JPEG is for photos.